In [7]:
from utils.visualization import *
from geometry.importers import STEPImporter
from solvers.frequency_domain import FrequencyDomainSolver
from rom.reduction import ModelOrderReduction
from analytical.rectangular_waveguide import RWGAnalytical
from ngsolve.webgui import Draw # must import Draw, otherwise may run into problems showing mesh
from utils.visualization import (
    plot_z_comparison, 
    plot_s_comparison, 
    plot_all_parameters,
    plot_eigenfrequencies,
    ParameterPlotter,
    EigenfrequencyPlotter
)
%matplotlib widget
plt.rcParams['figure.dpi'] = 100

## 1. Define Geometry

Create a circular waveguide with specified dimensions and mesh parameters.

In [12]:
class Models:
    def rwg(self, maxh=0.04):
        geo = STEPImporter(r"../rwg/rectangular_waveguide.step", auto_build=False)
        geo.build()
        geo.name_solids()
        geo.generate_mesh(maxh)
        return geo
    def rwg_split(self, maxh=0.04):
        geo = STEPImporter(r"../rwg/rectangular_waveguide.step", auto_build=False)
        geo.add_splitting_plane_at_z(0.05)
        geo.add_splitting_plane_at_z(0.10)
        geo.split()
        geo.show_split_preview()
        geo.build()
        geo.name_solids()

        geo.generate_mesh(maxh=0.03) # after naming solids, must generate mesh but avoid rebuilding
        return geo
    def cwg(self, maxh=0.04):
        geo = STEPImporter(r"circular_waveguide.step", auto_build=False)
        geo.build()
        geo.name_solids()
        geo.generate_mesh(maxh)
        return geo
    def cwg_split(self, maxh=0.04):
        geo = STEPImporter(r"circular_waveguide.step", auto_build=False)
        geo.add_splitting_plane_at_z(0.075)
        geo.add_splitting_plane_at_z(0.15)
        geo.split()

        geo.show_split_preview()
        geo.build()
        geo.name_solids()

        geo.generate_mesh(maxh=0.04) # after naming solids, must generate mesh but avoid rebuilding
        return geo

In [13]:
models = Models()
geo = models.cwg()
geo.show()

Single solid: port1 and port2 assigned


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

In [17]:
fmin, fmax, nsamples = 0.1, 1.5, 30
fds = FrequencyDomainSolver(geo, order=3)


Structure Topology
Type: Single structure
Domains (1): ['cell_1']
Total Ports (2): ['port1', 'port2']

Domain-Port Mapping:
  cell_1: ['port1 (external, input)', 'port2 (external, output)']


In [ ]:
fom = fds.solve(fmin, fmax, nsamples, solver_type='iterative')


Assembling Matrices...

Solving port eigenmodes...

	Calculating Port Eigenmodes...
	  Mode source: analytic
	  Polarization angle: 0.0°
	  Requested modes per port: 1
	------------------------------------------------------------
	  port2: circular (fit error: 0.0000)
	    R=0.150001
	  port1: circular (fit error: 0.0000)
	    R=0.150001
	  Precomputing boundary mass matrices (once per port)...
	    Done for 2 port(s)
	port2 mode 0: TE_11 (cos), kc=12.2745, σ=-1
	port1 mode 0: TE_11 (cos), kc=12.2745, σ=+1
	------------------------------------------------------------
	Total modes: 2

--- Assembling Per-Domain Matrices ---

Domain: cell_1
  FES ndof: 37428
  K shape: (37428, 37428), nnz: 3922512
  M shape: (37428, 37428), nnz: 3922512
  B shape: (37428, 2)

--- Assembling Global Matrices (Coupled System) ---
  Global FES ndof: 37428
  K_global shape: (37428, 37428), nnz: 3922512
  M_global shape: (37428, 37428), nnz: 3922512
  B_global shape: (37428, 2)

Frequency Domain Solve Configur

In [ ]:
fom.plo